# 03 — Transformación, Validación y Calidad (Persona 2)

Este notebook implementa las **Fases 3, 4 y 5** del pipeline ETL:
- Lectura de `data/bronze/` con **optimizaciones Spark** (5 obligatorias)
- Aplicación de **10 transformaciones avanzadas** (`apply_transformations`)
- Evaluación de **13 reglas de calidad** (`apply_quality_rules`)
- Escritura de `data/silver/`, `quality_rejected_records` y `quality_metrics_summary`

**Persona 2** | Fases 3-5 | 40 puntos

In [ ]:
%pip install -q pyspark pyyaml

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

from utils import load_config, get_spark_session
from transformations import apply_transformations
from quality_rules import apply_quality_rules, build_quality_metrics

config = load_config("config/etl_config.yaml")
spark = get_spark_session(config)
spark.sparkContext.setLogLevel("WARN")

process_id = config.get("process_id", "")
bronze_path = config["paths"]["bronze"]
silver_path = config["paths"]["silver"]
audit_path  = config["paths"]["audit"]

print("Proyecto:", PROJECT_ROOT)
print("process_id:", process_id)
print("bronze:", bronze_path)
print("silver:", silver_path)

## 1. Lectura de bronze/ con Optimizaciones Spark

Se aplican las 5 optimizaciones obligatorias:
1. **Esquema explícito (StructType)** — construido desde `canonical_schema_trips.json`
2. **Lectura selectiva de columnas** — solo las columnas necesarias
3. **Predicate pushdown** — `.filter()` antes de `.select()`
4. **Partition pruning** — filtro por `service_type` al leer particiones
5. **coalesce(4)** — aplicado en `apply_transformations` antes de escribir silver

In [ ]:
import json
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    IntegerType, TimestampType, LongType,
)

# Optimizacion #1: construir StructType explícito desde canonical_schema_trips.json
_TYPE_MAP = {
    "string": StringType(), "double": DoubleType(), "integer": IntegerType(),
    "timestamp": TimestampType(), "long": LongType(),
}
with open("metadata/canonical_schema_trips.json", "r") as f:
    canonical_meta = json.load(f)

BRONZE_SCHEMA = StructType([
    StructField(field["name"], _TYPE_MAP.get(field["type"], StringType()), field["nullable"])
    for field in canonical_meta["fields"]
])
print("Esquema explícito construido:", [f.name for f in BRONZE_SCHEMA.fields])

In [ ]:
from pyspark.sql.functions import col

# Columnas necesarias para transformaciones (lectura selectiva - Optimizacion #2)
NEEDED_COLS = [
    "trip_id", "service_type", "vendor_id",
    "pickup_datetime", "dropoff_datetime",
    "passenger_count", "trip_distance",
    "pickup_location_id", "dropoff_location_id",
    "payment_type", "fare_amount", "extra_amount", "mta_tax",
    "tip_amount", "tolls_amount", "total_amount",
    "congestion_surcharge", "airport_fee", "improvement_surcharge",
    "year", "month", "source_file", "ingestion_timestamp", "quality_status",
]

# Optimizacion #3: Predicate pushdown — filter ANTES de select
# Optimizacion #4: Partition pruning — filtrar por service_type al leer
df_bronze = (
    spark.read
    .schema(BRONZE_SCHEMA)           # Optimizacion #1
    .parquet(bronze_path)
    .filter(col("service_type").isin("yellow", "green", "fhvhv"))  # Optimizacion #3 + #4
    .select(*NEEDED_COLS)            # Optimizacion #2
)

print(f"Registros leidos de bronze: {df_bronze.count()}")
print(f"Columnas seleccionadas: {len(df_bronze.columns)}")
df_bronze.groupBy("service_type", "year", "month").count().orderBy("service_type","year","month").show()

## 2. Aplicación de Transformaciones Avanzadas

`apply_transformations(df)` agrega: `trip_duration_minutes`, `average_speed_mph`, `fare_per_mile`, `tip_percentage`, `is_airport_trip`, `is_suspicious_trip`, `processing_date` y normaliza montos a 2 decimales.

In [ ]:
df_transformed = apply_transformations(df_bronze)

print("=== Schema después de transformaciones ===")
df_transformed.printSchema()
print(f"\nTotal columnas: {len(df_transformed.columns)}")
print(f"Registros (post dedup): {df_transformed.count()}")

In [ ]:
# Plan de ejecucion Spark — evidencia de optimizaciones (Optimizacion #1-#4)
print("=== PLAN FISICO DE SPARK (evidencia de optimizaciones) ===")
df_transformed.explain(mode="formatted")

### Verificación de `is_suspicious_trip`

Se muestran 5 ejemplos de viajes sospechosos y 5 no sospechosos para validar la lógica.

In [ ]:
verify_cols = [
    "trip_id", "service_type", "trip_distance", "total_amount",
    "fare_amount", "trip_duration_minutes", "average_speed_mph",
    "tip_percentage", "pickup_datetime", "dropoff_datetime",
    "is_suspicious_trip",
]

print("=== Viajes SOSPECHOSOS (is_suspicious_trip = True) — primeros 5 ===")
df_transformed.filter(col("is_suspicious_trip") == True).select(*verify_cols).show(5, truncate=35)

print("=== Viajes NORMALES (is_suspicious_trip = False) — primeros 5 ===")
df_transformed.filter(col("is_suspicious_trip") == False).select(*verify_cols).show(5, truncate=35)

n_suspicious = df_transformed.filter(col("is_suspicious_trip") == True).count()
n_total = df_transformed.count()
print(f"\nViajes sospechosos: {n_suspicious} / {n_total} ({100*n_suspicious/n_total:.2f}%)")

## 3. Aplicación de Reglas de Calidad

`apply_quality_rules(df, process_id)` evalúa las 13 reglas y separa registros válidos e inválidos.
Un mismo registro puede violar múltiples reglas, generando una fila en `quality_rejected_records` por cada violación.

In [ ]:
df_valid, df_rejected = apply_quality_rules(df_transformed, process_id)

n_valid    = df_valid.count()
n_rejected = df_rejected.count()
print(f"Registros VALIDOS (silver):    {n_valid}")
print(f"Filas en rejected_records:     {n_rejected}  (puede ser > rechazados reales si hay multi-violacion)")
print()

print("=== Rechazos por regla ===")
df_rejected.groupBy("rejection_rule").count().orderBy("count", ascending=False).show(20)

## 4. Escritura de data/silver/ y tablas de auditoría de calidad

**Optimización #5** ya aplicada en `apply_transformations`: `.coalesce(4)` antes de escribir silver.

In [ ]:
# Escritura de data/silver/ particionada por service_type / year / month
# df_valid ya viene con coalesce(4) aplicado desde apply_transformations
silver_output = str(PROJECT_ROOT / silver_path)

(
    df_valid
    .write
    .mode("overwrite")
    .partitionBy("service_type", "year", "month")
    .parquet(silver_output)
)

print(f"Silver escrito en: {silver_output}")
print("Particiones: service_type / year / month")

# Verificación inmediata: contar registros por partición
df_silver_check = spark.read.parquet(silver_output)
print(f"\nTotal registros en silver: {df_silver_check.count()}")
df_silver_check.groupBy("service_type", "year", "month").count().orderBy("service_type", "year", "month").show()

In [ ]:
# Escritura de data/audit/quality_rejected_records/
rejected_output = str(PROJECT_ROOT / audit_path / "quality_rejected_records")

(
    df_rejected
    .write
    .mode("overwrite")
    .parquet(rejected_output)
)

print(f"quality_rejected_records escrito en: {rejected_output}")
print(f"Total filas en quality_rejected_records: {df_rejected.count()}")
print()

# Verificación: schema de la tabla de rechazos
print("=== Schema de quality_rejected_records ===")
df_rejected.printSchema()

In [ ]:
# Construcción y escritura de data/audit/quality_metrics_summary/
df_quality_metrics = build_quality_metrics(df_transformed, df_valid, df_rejected, process_id)

metrics_output = str(PROJECT_ROOT / audit_path / "quality_metrics_summary")

(
    df_quality_metrics
    .write
    .mode("overwrite")
    .parquet(metrics_output)
)

print(f"quality_metrics_summary escrito en: {metrics_output}")
print()

print("=== quality_metrics_summary ===")
df_quality_metrics.show(truncate=False)

## 5. Verificación Final del Pipeline

Criterios de aceptación a comprobar:
1. Silver no contiene `trip_distance <= 0`
2. Silver no contiene `total_amount <= 0`
3. `quality_percentage` > 0 para todos los grupos
4. Columnas de silver = columnas de bronze + campos calculados

In [ ]:
from pyspark.sql.functions import col as _col

df_silver_final = spark.read.parquet(silver_output)

# Criterio 1: no hay trip_distance <= 0 en silver
n_zero_dist = df_silver_final.filter(_col("trip_distance") <= 0).count()
print(f"[1] Registros con trip_distance <= 0 en silver: {n_zero_dist}  (debe ser 0)")

# Criterio 2: no hay total_amount <= 0 en silver
n_neg_total = df_silver_final.filter(_col("total_amount") <= 0).count()
print(f"[2] Registros con total_amount <= 0 en silver: {n_neg_total}  (debe ser 0)")

# Criterio 3: quality_percentage en rango válido
print("\n[3] quality_percentage por servicio/mes:")
spark.read.parquet(metrics_output).select(
    "service_type", "year", "month",
    "total_records", "valid_records", "rejected_records", "quality_percentage"
).orderBy("service_type", "year", "month").show(truncate=False)

# Criterio 4: columnas de silver
silver_cols = df_silver_final.columns
bronze_cols = df_bronze.columns
nuevas_cols = [c for c in silver_cols if c not in bronze_cols]
print(f"\n[4] Columnas en bronze: {len(bronze_cols)}")
print(f"    Columnas en silver:  {len(silver_cols)}")
print(f"    Columnas añadidas por transformaciones: {nuevas_cols}")

# Resumen ejecutivo
print("\n" + "="*60)
print("RESUMEN EJECUTIVO — PERSONA 2 (Fases 3, 4 y 5)")
print("="*60)
total_bronze  = df_bronze.count()
total_silver  = df_silver_final.count()
total_rejected_unique = df_rejected.select("trip_id").distinct().count()
print(f"  Registros leídos de bronze:          {total_bronze}")
print(f"  Registros escritos en silver:        {total_silver}")
print(f"  Viajes únicos rechazados por calidad:{total_rejected_unique}")
print(f"  Filas en quality_rejected_records:   {df_rejected.count()} (multiviol.)")
print(f"  Tasa de retención global:            {100*total_silver/total_bronze:.2f}%")
print("="*60)
print("Pipeline completado. Tablas de auditoría generadas en data/audit/")